# Task
Integrate the `lora_finetuning.py` and `lora_finetuning_test.py` scripts into Google Colab. Adapt both scripts to run within Jupyter cells by replacing `argparse` with direct variable assignments for `data_dir` and `output_dir`. Ensure Google Drive is mounted, and set `data_dir` to "/content/drive/MyDrive/PyPilot/data/leetcode" and `output_dir` to "/content/drive/MyDrive/PyPilot/outputs/qwen-lora". Additionally, make sure all necessary functions from `code_execution.py` are available for the test script.

## Mount Google Drive

### Subtask:
Add code to mount Google Drive in the Colab environment to access `/content/drive/MyDrive/`.


**Reasoning**:
To mount Google Drive, I need to import the `drive` module from `google.colab`, call its `mount()` method with `/content/drive`, and then verify the mount by listing the contents of `/content/drive/MyDrive`.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify that Google Drive is successfully mounted
!ls /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 00001.vcf
'11 13 20 - Checklist.gdoc'
'2020 Scioly Elections Prep'
'2022W 136-2 03 Uncertainties.docx'
'2026 AI Resume.docx'
'2026 AI Resume.pdf'
'2026 Odd-jobs Resume.gdoc'
 4496323488_1.pdf
'5 22 therapy .gdoc'
'5 31 therapy.gdoc'
'8 5.gdoc'
'A6 FINAL720p.mov'
 Accounts.dmg
 Affirmations.gdoc
 Alpaca_Data.ipynb
 alpaca_get_intraday_full.ipynb
"Ang's Mock Interview 1.gdoc"
'Apartment Furniture 2024.gsheet'
'Apartment Shopping.gsheet'
'AP Calculus BC Unit 5 Study guide.gdoc'
'AP Calculus BC Unit 6 Study Guide and Plan.gdoc'
'ARIMA-GARCH Volatility Forecasting.ipynb'
'Backtester rough diagram (1).gdraw'
'Backtester rough diagram.gdraw'
'Backtester Specification.gdraw'
'Backtesting Framework.ipynb'
 Backtesting.gdoc
'Bday note .gdoc'
'BERT Finetuning - CS 461 HW 2.ipynb'
'Beyonders 3 - Chasing the Prophecy.pdf'
'Big Goals (5 and 10 Years).gdoc'
'Bio and profil

**Reasoning**:
The Google Drive has been successfully mounted. Now, to run the `lora_finetuning.py` script, I need to install all the required Python packages mentioned in the script's `Requirements` section.



In [ ]:
!pip install torch transformers datasets peft accelerate bitsandbytes trl

**Reasoning**:
Now that the necessary libraries are installed, I will adapt the `lora_finetuning.py` script for the Colab environment. This involves taking the original script code, replacing the `argparse` argument parsing with direct variable assignments for `data_dir` and `output_dir` as specified, and then calling the `main` function directly.



In [ ]:
from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Tuple

import torch
from datasets import load_from_disk
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl.trainer import SFTConfig, SFTTrainer


MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Qwen chat tokens (match training/inference format)
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"


def format_example(example: dict) -> dict:
    """Step 1: just build the text strings, no tokenizer needed."""
    instruction = (example.get("query") or "").strip()
    instruction = instruction.replace("(use the provided format with backticks)", "")
    instruction = instruction.replace("and enclose your code within delimiters.", "")
    instruction = instruction.rstrip()
    instruction += "\n\nRespond with only the Python code. No explanations, no markdown."

    user_prompt = f"{CHAT_USER}{instruction}{CHAT_END}\n{CHAT_ASSISTANT}"
    response = example["completion"] + CHAT_END

    return {"prompt": user_prompt, "response": response}


def tokenize_example(example: dict, tokenizer, max_seq_length: int) -> dict:
    """Step 2: tokenize and build masked labels. Called after tokenizer is loaded."""
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(example["response"], add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + response_ids)[:max_seq_length]
    labels = ([-100] * len(prompt_ids) + response_ids)[:max_seq_length]

    return {"input_ids": input_ids, "labels": labels}



def load_model_and_tokenizer(
    model_id: str,
    use_4bit: bool = True,
    use_flash_attention: bool = False,
):
    """Load Qwen2.5-Coder with optional 4-bit quantization."""

    # Quantization config for memory efficiency
    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        padding_side="right",
    )



    # Set pad token if not present
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id



    # Model loading kwargs
    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16,
        "device_map": "auto",
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    if use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.config.pad_token_id = tokenizer.eos_token_id

    # Prepare model for k-bit training if using quantization
    if use_4bit:
        model = prepare_model_for_kbit_training(model)

    return model, tokenizer


def check_bf16_support() -> Tuple[bool, str]:
    """
    Check if the current setup supports bfloat16.

    Returns:
        Tuple of (supports_bf16: bool, info_message: str)
    """
    info_lines = []

    # Check CUDA availability
    if not torch.cuda.is_available():
        return False, "CUDA is not available"

    info_lines.append(f"CUDA available: {torch.cuda.is_available()}")
    info_lines.append(f"CUDA version: {torch.version.cuda}")
    info_lines.append(f"PyTorch version: {torch.__version__}")

    # Check GPU info
    try:
        device_count = torch.cuda.device_count()
        info_lines.append(f"GPU count: {device_count}")

        for i in range(device_count):
            gpu_name = torch.cuda.get_device_name(i)
            device_capability = torch.cuda.get_device_capability(i)
            info_lines.append(f"GPU {i}: {gpu_name} (Compute Capability: {device_capability[0]}.{device_capability[1]})")

            # Check if compute capability supports bf16 (Ampere 8.0+ or Ada Lovelace 8.9+)
            if device_capability[0] >= 8:
                info_lines.append(f"  -> GPU {i} architecture supports bf16 (Ampere/Ada Lovelace)")
            else:
                info_lines.append(f"  -> GPU {i} architecture may not support bf16 (pre-Ampere)")

        # Try to create a bf16 tensor to test actual support
        try:
            test_tensor = torch.tensor([1.0], dtype=torch.bfloat16, device="cuda:0")
            info_lines.append("bf16 tensor creation test: SUCCESS")
            supports_bf16 = True
        except Exception as e:
            info_lines.append(f"bf16 tensor creation test: FAILED - {e}")
            supports_bf16 = False

    except Exception as e:
        info_lines.append(f"Error checking GPU: {e}")
        supports_bf16 = False

    info_message = "\n".join(info_lines)
    return supports_bf16, info_message


def create_lora_config() -> LoraConfig:
    """Create LoRA configuration for Qwen2.5-Coder.

    Matches the notebook implementation:
    - rank r=4 (notebook uses r=4)
    - alpha=r for scaling factor of 1.0 (notebook does simple addition: original + lora)
    - Applied to attention and MLP projection layers
    """
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=4,                           # LoRA rank (matching notebook: r=4)
        lora_alpha=16,                  # LoRA alpha = rank for scaling factor of 1.0 (matches notebook's simple addition)
        lora_dropout=0.1,             # Dropout for LoRA layers
        target_modules=[               # Qwen2.5 attention modules
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        bias="none",
    )


def main(args):
    print(f"Loading dataset from {args.data_dir}...")
    dataset = load_from_disk(args.data_dir)
    row = dataset["train"][0]
    print("TRAIN keys:", row.keys())
    print("\nQUERY head:\n", (row.get("query","")[:300]))
    print("\nCOMPLETION head:\n", (row.get("completion","")[:300]))

    print(f"Train samples: {len(dataset['train'])}")
    print(f"Test samples: {len(dataset['test'])}")

    # Format dataset
    print("Formatting dataset...")
    train_dataset = dataset["train"].map(
        format_example,
        remove_columns=dataset["train"].column_names,
        desc="Formatting train",
    )
    eval_dataset = dataset["test"].map(
        format_example,
        remove_columns=dataset["test"].column_names,
        desc="Formatting eval",
    )

    # Optionally limit dataset size for faster iteration
    if args.max_train_samples:
        train_dataset = train_dataset.select(range(min(args.max_train_samples, len(train_dataset))))
    if args.max_eval_samples:
        eval_dataset = eval_dataset.select(range(min(args.max_eval_samples, len(eval_dataset))))

    print(f"Training on {len(train_dataset)} samples, evaluating on {len(eval_dataset)} samples")

    # --- Paste this cell AFTER your dataset formatting cell, BEFORE training ---

    import numpy as np
    from transformers import AutoTokenizer

    MAX_SEQ_LENGTH = 2048  # Must match your training config

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    # Assumes train_dataset already has the "text" field from format_example()
    # If you haven't run formatting yet, do it first:
    # train_dataset = dataset["train"].map(format_example, remove_columns=dataset["train"].column_names)

    lengths = []
    for example in train_dataset:
        # Concatenate prompt + completion to get full sequence length
        full_text = example["prompt"] + example["response"]
        token_ids = tokenizer(full_text, truncation=False, add_special_tokens=False)["input_ids"]
        lengths.append(len(token_ids))

    lengths = np.array(lengths)

    print(f"Total examples: {len(lengths)}")
    print(f"Max seq length setting: {MAX_SEQ_LENGTH}")
    print(f"---")
    print(f"Min tokens:    {lengths.min()}")
    print(f"Median tokens: {int(np.median(lengths))}")
    print(f"Mean tokens:   {int(np.mean(lengths))}")
    print(f"P90 tokens:    {int(np.percentile(lengths, 90))}")
    print(f"P95 tokens:    {int(np.percentile(lengths, 95))}")
    print(f"P99 tokens:    {int(np.percentile(lengths, 99))}")
    print(f"Max tokens:    {lengths.max()}")
    print(f"---")
    truncated = (lengths > MAX_SEQ_LENGTH).sum()
    print(f"Truncated (>{MAX_SEQ_LENGTH}): {truncated}/{len(lengths)} ({100*truncated/len(lengths):.1f}%)")

    # If >10% are truncated, the <|im_end|> at the end is being cut off
    # for those examples, so the model never learns to stop on them.
    if truncated / len(lengths) > 0.10:
        print(f"\n⚠️  {100*truncated/len(lengths):.0f}% of examples are truncated!")
        print(f"   The trailing <|im_end|> is being cut off for these samples.")
        print(f"   Consider increasing max_seq_length to {int(np.percentile(lengths, 95))} (P95)")
        print(f"   or to {int(np.percentile(lengths, 99))} (P99).")
    else:
        print(f"\n✅ Truncation looks fine — only {100*truncated/len(lengths):.1f}% affected.")

    # Load model and tokenizer
    print(f"Loading model: {MODEL_ID}...")
    model, tokenizer = load_model_and_tokenizer(
        MODEL_ID,
        use_4bit=args.use_4bit,
        use_flash_attention=args.use_flash_attention,
    )

    tokenizer.model_max_length = args.max_seq_length

    from functools import partial
    tok_fn = partial(tokenize_example, tokenizer=tokenizer, max_seq_length=args.max_seq_length)

    train_dataset = train_dataset.map(tok_fn, remove_columns=["prompt", "response"], desc="Tokenizing train")
    eval_dataset  = eval_dataset.map(tok_fn, remove_columns=["prompt", "response"], desc="Tokenizing eval")


    # Apply LoRA
    print("Applying LoRA adapters...")
    lora_config = create_lora_config()
    model = get_peft_model(model, lora_config)

    # Verify that base weights are frozen (PEFT should do this automatically)
    # Count trainable vs total parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"Total parameters: {total_params:,}")

    model.print_trainable_parameters()

    # Check bf16 support
    print("\n" + "="*60)
    print("Checking bf16 support...")
    print("="*60)
    supports_bf16, bf16_info = check_bf16_support()
    print(bf16_info)
    print("="*60 + "\n")

    # Determine mixed precision settings
    if supports_bf16:
        print("Using bfloat16 mixed precision training")
        use_bf16 = True
        use_fp16 = False
    elif torch.cuda.is_available():
        print("bf16 not supported, falling back to float16 mixed precision")
        use_bf16 = False
        use_fp16 = True
    else:
        print("No GPU available, using full precision (fp32)")
        use_bf16 = False
        use_fp16 = False

    # SFTConfig with completion-only loss (replaces TrainingArguments + DataCollatorForCompletionOnlyLM)
    output_dir = Path(args.output_dir)

    # Build config kwargs
    config_kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": args.epochs,
        "per_device_train_batch_size": args.batch_size,
        "per_device_eval_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.gradient_accumulation_steps,
        "learning_rate": args.learning_rate,
        "weight_decay": 0.01,
        "warmup_ratio": 0.03,
        "lr_scheduler_type": "cosine",
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": args.eval_steps,
        "save_strategy": "steps",
        "save_steps": args.save_steps,
        "save_total_limit": 3,
        "load_best_model_at_end": True, # For early stopping
        "metric_for_best_model": "eval_loss", # For early stopping
        "gradient_checkpointing": args.gradient_checkpointing,
        "optim": "paged_adamw_8bit" if args.use_4bit else "adamw_torch",
        "report_to": "none",
        "push_to_hub": False,
        "max_grad_norm": 1.0,
        "dataloader_num_workers": 0,
        "remove_unused_columns": False,   # IMPORTANT: keep input_ids/labels
        # removed: completion_only_loss, max_length
    }

    # Add mixed precision settings
    if use_bf16:
        config_kwargs["bf16"] = True
        config_kwargs["fp16"] = False
    elif use_fp16:
        config_kwargs["bf16"] = False
        config_kwargs["fp16"] = True
    else:
        config_kwargs["bf16"] = False
        config_kwargs["fp16"] = False

    # Add gradient checkpointing kwargs if enabled
    if args.gradient_checkpointing:
        config_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    sft_config = SFTConfig(**config_kwargs)



    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )




    # Debug: check if completion masking is working
    sample = trainer.get_train_dataloader()
    batch = next(iter(sample))
    print("Keys in batch:", batch.keys())
    print("Has completion_mask:", "completion_mask" in batch)

    # Check if labels are masked (should have -100 for prompt tokens)
    labels = batch["labels"][0]
    num_masked = (labels == -100).sum().item()
    num_total = labels.shape[0]
    print(f"Masked tokens: {num_masked}/{num_total} ({100*num_masked/num_total:.1f}%)")

    # If masked is 0% or very low, completion_only_loss isn't working
    # Train
    print("Starting training...")
    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    # Save final model
    final_model_path = output_dir / "final"
    print(f"Saving final model to {final_model_path}...")
    trainer.save_model(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))

    print("Training complete!")


# Replace argparse with direct assignments for Colab
class Args:
    def __init__(self):
        self.data_dir = "/content/drive/MyDrive/PyPilot/data/leetcode"
        self.output_dir = "/content/drive/MyDrive/PyPilot/outputs/qwen-lora"
        self.max_train_samples = None
        self.max_eval_samples = None
        self.use_4bit = True
        self.use_flash_attention = False
        self.epochs = 1
        self.batch_size = 2
        self.gradient_accumulation_steps = 8
        self.learning_rate = 1e-5
        self.max_seq_length = 3072
        self.gradient_checkpointing = True
        self.eval_steps = 50
        self.save_steps = 50
        self.resume_from_checkpoint = None

args = Args()
main(args)

Loading dataset from /content/drive/MyDrive/PyPilot/data/leetcode...
TRAIN keys: dict_keys(['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'])

QUERY head:
 You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
Given an array of integers nums and an integer target, return indices of the two numbers such that 

COMPLETION head:
 class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        d = {}
        for i, x in enumerate(nums):
            if (y := target - x) in d:
                return [d[y], i]
            d[x] = i

Train samples: 2641
Test samples: 228
Formatting dataset...


Formatting train:   0%|          | 0/2641 [00:00<?, ? examples/s]

Formatting eval:   0%|          | 0/228 [00:00<?, ? examples/s]

Training on 2641 samples, evaluating on 228 samples


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Truncating eval dataset:   0%|          | 0/228 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Keys in batch: dict_keys(['input_ids', 'labels', 'attention_mask'])
Has completion_mask: False
Masked tokens: 487/661 (73.7%)
Starting training...


Step,Training Loss,Validation Loss
50,0.324323,0.532164
100,0.279988,0.503291
150,0.276826,0.499640


Saving final model to /content/drive/MyDrive/PyPilot/outputs/qwen-lora/final...
Training complete!


In [ ]:
!python -c "import trl; print(trl.__version__)"
!python -c "from trl.trainer import DataCollatorForCompletionOnlyLM; print('Found in trl.trainer')" 2>/dev/null || echo "Not in trl.trainer"
!python -c "from trl.data_utils import DataCollatorForCompletionOnlyLM; print('Found in trl.data_utils')" 2>/dev/null || echo "Not in trl.data_utils"
!python -c "import trl; print([x for x in dir(trl) if 'Collator' in x or 'collator' in x])"


0.29.0
Not in trl.trainer
Not in trl.data_utils
[]


In [ ]:
from trl import DataCollatorForCompletionOnlyLM

ImportError: cannot import name 'DataCollatorForCompletionOnlyLM' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [ ]:
!grep -r "CompletionOnly\|completion_only\|response_template" /usr/local/lib/python3.12/dist-packages/trl/ --include="*.py" -l

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_config.py
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py


In [ ]:
!grep -rn "class.*Collator" /usr/local/lib/python3.12/dist-packages/trl/ --include="*.py"

/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:94:class DataCollatorForPreference(DataCollatorMixin):
/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:194:class DataCollatorForVisionPreference(DataCollatorMixin):
/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:127:class DataCollatorForPreference(DataCollatorMixin):
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:91:class DataCollatorForLanguageModeling(DataCollatorMixin):
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:259:class DataCollatorForVisionLanguageModeling(DataCollatorMixin):
/usr/local/lib/python3.12/dist-packages/trl/experimental/kto/kto_trainer.py:450:                "max_length or a processing_class must be specified when using the default DPODataCollatorWithPadding"
/usr/local/lib/python3.12/dist-packages/trl/experimental/utils.py:46:class DPODataCollatorWithPadding:
/usr/local/lib/python3.12/dist-packages/trl/experimental/u

In [ ]:
!grep -n "completion_only\|response_template" /usr/local/lib/python3.12/dist-packages/trl/trainer/sft_config.py
!echo "==="
!grep -n "completion_only\|response_template" /usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py

90:        completion_only_loss (`bool`, *optional*):
214:    completion_only_loss: bool | None = field(
===
100:    - `"labels"`: Tensor of labels, padded to the maximum length of the batch. If `completion_only_loss` is set to
111:        completion_only_loss (`bool`, *optional*, defaults to `True`):
159:    completion_only_loss: bool = True
183:        if self.completion_only_loss and "completion_mask" in examples[0]:
194:            if self.completion_only_loss and "completion_mask" in examples[0]:
218:        if self.completion_only_loss and "completion_mask" in examples[0]:
288:        completion_only_loss (`bool`, *optional*, defaults to `False`):
339:    completion_only_loss: bool = False  # default not used in practice; SFTTrainer always passes the relevant value
346:            if self.completion_only_loss:
348:                    "The `completion_only_loss` argument is not supported for language modeling datasets."
463:        if self.completion_only_loss:
811:        if args

In [ ]:
from transformers import DataCollatorForLanguageModeling

## Integrate `code_execution.py` utilities

### Subtask:
Copy the content of `code_execution.py` (from cell `Iog2oKjM1nsH`) into a new code cell to make its functions directly available for the test script.

In [ ]:
import ast
import subprocess
import sys
import tempfile
import os
from pathlib import Path
from typing import Tuple, Optional

def check_compilation(code: str) -> Tuple[bool, Optional[str]]:
    """
    Check if Python code compiles without syntax errors.

    Args:
        code: Python code string to check

    Returns:
        Tuple of (compiles: bool, error_message: Optional[str])
    """
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg} at line {e.lineno}"
    except Exception as e:
        return False, f"ParseError: {str(e)}"


def extract_code_from_completion(completion: str, starter_code: str = "") -> str:
    """
    Extract Python code from model completion.

    This function attempts to extract the actual code from the model's output,
    which might include markdown code blocks, explanations, etc.

    Args:
        completion: Model's raw completion text
        starter_code: Optional starter code to prepend

    Returns:
        Extracted Python code string
    """
    # Strip Qwen-style chat markers and special tokens if they leak into the output
    for marker in (
        "<|im_start|>system",
        "<|im_start|>user",
        "<|im_start|>assistant",
        "<|im_start|>",
        "<|im_end|>",
        "<|endoftext|>",
    ):
        completion = completion.replace(marker, "")

    # Truncate at Qwen FIM/file tokens and common failure modes
    # "FRINGEMENT" and "{lng" were observed in diagnostic output
    stop_tokens = [
        "<|file_sep|>",
        "<|fim_prefix|>",
        "<|fim_suffix|>",
        "<|fim_middle|>",
        "<|repo_name|>",
        "FRINGEMENT",
        "\nuser\n", # often hallucinated start of next turn
        "\n{lng",
    ]

    for stop_token in stop_tokens:
        if stop_token in completion:
            completion = completion[:completion.find(stop_token)]

    # Remove markdown code blocks if present
    if "```python" in completion:
        # Extract code between ```python and ```
        start_idx = completion.find("```python") + len("```python")
        end_idx = completion.find("```", start_idx)
        if end_idx != -1:
            completion = completion[start_idx:end_idx].strip()
        else:
             # If no closing tick, take everything after start
            completion = completion[start_idx:].strip()
    elif "```" in completion:
        # Generic code block
        start_idx = completion.find("```") + 3
        end_idx = completion.find("```", start_idx)
        if end_idx != -1:
            completion = completion[start_idx:end_idx].strip()
        else:
            completion = completion[start_idx:].strip()

    # Post-process: Add typing imports if type annotations are used
    extracted_code = completion.strip()

    # Check if code uses type annotations that need imports
    needs_typing = any(annotation in extracted_code for annotation in
                       ['List[', 'Dict[', 'Tuple[', 'Optional[', 'Any', 'Union[', 'Set['])

    if needs_typing and 'from typing import' not in extracted_code:
        # Determine which typing imports are needed
        needed_imports = []
        if 'List[' in extracted_code:
            needed_imports.append('List')
        if 'Dict[' in extracted_code:
            needed_imports.append('Dict')
        if 'Tuple[' in extracted_code:
            needed_imports.append('Tuple')
        if 'Optional[' in extracted_code:
            needed_imports.append('Optional')
        if 'Union[' in extracted_code:
            needed_imports.append('Union')
        if 'Set[' in extracted_code:
            needed_imports.append('Set')
        if 'Any' in extracted_code and 'Any' not in needed_imports:
            needed_imports.append('Any')

        if needed_imports:
            typing_import = f"from typing import {', '.join(needed_imports)}\n"
            extracted_code = typing_import + extracted_code

    # Add common stdlib imports if used but not imported
    stdlib_imports = []
    if 'defaultdict' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import defaultdict, deque, Counter')
    elif 'deque' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import deque')
    elif 'Counter' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import Counter')

    if 'heappush' in extracted_code or 'heappop' in extracted_code:
        if 'from heapq' not in extracted_code and 'import heapq' not in extracted_code:
            stdlib_imports.append('from heapq import heappush, heappop, heapify')

    if ' inf' in extracted_code or '(inf' in extracted_code or '[inf' in extracted_code:
        if 'inf = ' not in extracted_code and 'from math import inf' not in extracted_code:
            stdlib_imports.append('from math import inf')

    if stdlib_imports:
        extracted_code = '\n'.join(stdlib_imports) + '\n' + extracted_code

    # Combine with starter code if provided
    if starter_code:
        # Extract imports from starter code that might be needed
        starter_lines = starter_code.split('\n')
        starter_imports = [line for line in starter_lines
                          if line.strip().startswith(('import ', 'from '))]

        # Check if extracted code already has a class or function definition
        # If so, don't prepend starter code (model provided complete solution)
        has_class_def = 'class ' in extracted_code
        has_func_def = 'def ' in extracted_code

        # Only prepend starter code if the model output doesn't have its own structure
        if not has_class_def and not has_func_def:
            # Check if we need to add starter imports
            if starter_imports:
                extracted_lines = extracted_code.split('\n')
                existing_imports = [line for line in extracted_lines
                                  if line.strip().startswith(('import ', 'from '))]
                for imp in starter_imports:
                    if imp not in existing_imports:
                        extracted_code = imp + '\n' + extracted_code

            # Prepend starter code since model only output function body
            if starter_code.strip() not in extracted_code:
                combined = starter_code + "\n" + extracted_code
                return combined

    return extracted_code


def run_tests(code: str, test_code: str, entry_point: str = "candidate", timeout: int = 10) -> Tuple[bool, Optional[str]]:
    """
    Execute code and run tests in a subprocess with timeout.

    The test_code should contain a check(candidate) function that tests the solution.
    We'll call check(entry_point) where entry_point is the function name from the dataset.

    Args:
        code: The solution code to test
        test_code: The test code containing check(candidate) function
        entry_point: The name of the function to test (default: "candidate")
        timeout: Maximum execution time in seconds

    Returns:
        Tuple of (tests_passed: bool, error_message: Optional[str])
    """
    # Combine code and tests
    # The test_code contains check(candidate), so we need to call it with the entry_point function
    full_code = code + "\n\n" + test_code + f"\n\n# Run the tests\ncheck({entry_point})"

    # Create a temporary file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(full_code)
        temp_file = f.name

    try:
        # Run the code in a subprocess with timeout
        result = subprocess.run(
            [sys.executable, temp_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=os.path.dirname(temp_file)
        )

        # Check if tests passed (exit code 0 typically means success)
        if result.returncode == 0:
            return True, None
        else:
            error_msg = result.stderr or result.stdout
            return False, error_msg[:500]  # Limit error message length

    except subprocess.TimeoutExpired:
        return False, f"Timeout after {timeout} seconds"
    except Exception as e:
        return False, f"Execution error: {str(e)}"
    finally:
        # Clean up temp file
        try:
            os.unlink(temp_file)
        except:
            pass

## Adapt `lora_finetuning_test.py`

### Subtask:
Adapt the `lora_finetuning_test.py` script for the Colab environment by replacing `argparse` with direct variable assignments for `model_path`, `data_dir`, etc., and then run the evaluation.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_from_disk
from tqdm import tqdm
import logging
from pathlib import Path
from typing import Dict

# The functions check_compilation, extract_code_from_completion, and run_tests
# are assumed to be defined in the previous cell or imported.


# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Qwen chat tokens (match training format)
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"


def build_model_prompt(row: dict, include_prelude: bool = False) -> str:
    # Use ONLY query — prompt field is test harness boilerplate
    user_content = (row.get("query") or "").strip()

    user_content = user_content.replace("(use the provided format with backticks)", "")
    user_content = user_content.replace("and enclose your code within delimiters.", "")
    user_content = user_content.rstrip()
    user_content += "\n\nRespond with only the Python code. No explanations, no markdown."

    return f"{CHAT_USER}{user_content}{CHAT_END}\n{CHAT_ASSISTANT}"


def load_lora_model(base_model_id: str, lora_model_path: str, use_4bit: bool = True):
    """
    Load the base model and apply LoRA adapters.

    Args:
        base_model_id: HuggingFace model identifier for the base model
        lora_model_path: Path to the finetuned LoRA adapters
        use_4bit: Whether to use 4-bit quantization

    Returns:
        Tuple of (model, tokenizer)
    """
    from transformers import BitsAndBytesConfig

    logger.info(f"Loading base model: {base_model_id}")

    # Quantization config for memory efficiency (same as training)
    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_id,
        trust_remote_code=True,
        padding_side="right",
    )

    # Set pad token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id


    # Load base model
    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **model_kwargs)

    # Load LoRA adapters
    logger.info(f"Loading LoRA adapters from: {lora_model_path}")
    model = PeftModel.from_pretrained(base_model, lora_model_path)
    model.config.pad_token_id = tokenizer.eos_token_id
    print(type(model))
    print(model.peft_config)

    # Merge adapters for faster inference (optional - comment out if you want to keep them separate)
    # logger.info("Merging LoRA adapters for faster inference...")
    # model = model.merge_and_unload()

    logger.info("Model loaded successfully")
    return model, tokenizer

def clean_code(text: str) -> str:
    # Keep only first class Solution block
    start = text.find("class Solution:")
    if start == -1:
        return text.strip()

    text = text[start:]

    # If another class starts, cut it
    second = text.find("\nclass Solution:", 10)
    if second != -1:
        text = text[:second]

    # Remove trailing partial word after return
    lines = text.splitlines()
    if lines:
        last = lines[-1]
        # If last line has no indentation but contains no valid Python structure
        if (
            not last.startswith(" ")
            and not last.startswith("\t")
            and "class" not in last
            and "def" not in last
            and "=" not in last
        ):
            lines = lines[:-1]

    return "\n".join(lines).strip()


def generate_code(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 256,   # lower is safer
    do_sample: bool = False,
    temperature: float = 0.2,
) -> str:

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # === STOP TOKENS ===
    eos_ids = []

    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            eos_ids.append(tid)

    # === BAN FIM / REPO TOKENS ===
    banned = [
        "<|fim_prefix|>",
        "<|fim_middle|>",
        "<|fim_suffix|>",
        "<|fim_pad|>",
        "<|repo_name|>",
    ]

    bad_words_ids = []
    for tok in banned:
        ids = tokenizer.encode(tok, add_special_tokens=False)
        if ids:
            bad_words_ids.append(ids)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=eos_ids,
        bad_words_ids=bad_words_ids,
    )

    # Remove None entries
    gen_kwargs = {k: v for k, v in gen_kwargs.items() if v is not None}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    # === HARD CUT ON STOP MARKERS ===
    for marker in [
        "<|im_end|>",
        "<|endoftext|>",
        "<|repo_name|>",
        "<|fim_prefix|>",
        "<|fim_middle|>",
        "<|fim_suffix|>",
        "<|fim_pad|>",
    ]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]

    # === KEEP ONLY FIRST class Solution BLOCK ===
    first = text.find("class Solution:")
    if first != -1:
        text = text[first:]
        second = text.find("\nclass Solution:", 10)
        if second != -1:
            text = text[:second]

    return text.strip()

def evaluate_on_dataset(
    model,
    tokenizer,
    dataset,
    max_samples: int = None,
    include_prelude: bool = True,
    max_new_tokens: int = 512,
    temperature: float = 0.0,
) -> Dict[str, float]:
    """
    Evaluate model on test dataset.

    Args:
        model: The language model
        tokenizer: The tokenizer
        dataset: Test dataset
        max_samples: Maximum number of samples to evaluate (None for all)
        include_prelude: Whether to include prelude in the prompt (default: True, matching training)
        max_new_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature for generation

    Returns:
        Dictionary with metrics: compile_rate, test_pass_rate, total_samples
    """
    test_split = dataset["test"]
    total_samples = len(test_split) if max_samples is None else min(max_samples, len(test_split))

    logger.info(f"Evaluating on {total_samples} samples from test set")

    compile_count = 0
    test_pass_count = 0
    total_evaluated = 0

    for idx in tqdm(range(total_samples), desc="Evaluating"):

        row = test_split[idx]

        # Build prompt using build_model_prompt function
        prompt = build_model_prompt(row, include_prelude=include_prelude)
        starter_code = row.get("starter_code", "")
        test_code = row.get("test", "")
        entry_point = row.get("entry_point", "candidate")
        task_id = row.get("task_id", f"task_{idx}")

        if idx < 5:
          print(f"Prompt: {prompt}")
          print(f"Starter Code: {starter_code}")

        if not prompt.strip() or not test_code:
            logger.warning(f"Skipping sample {idx}: missing query/prompt or test code")
            continue

        # Generate code
        try:
            generated_completion = clean_code(generate_code(
                model,
                tokenizer,
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
            ))
            if idx < 5:
                print(f"\nGenerated Completion: {generated_completion}\n")
        except Exception as e:
            logger.error(f"Error generating code for {task_id}: {e}")
            continue

        # Extract code from completion
        extracted_code = extract_code_from_completion(generated_completion, starter_code)

        # Check compilation
        full_solution = extracted_code

        compiles, compile_error = check_compilation(full_solution)

        if compiles:
            compile_count += 1
        else:
            logger.debug(f"Compilation failed for {task_id}: {compile_error}")

        # Run tests (only if code compiles)
        tests_passed = False
        if compiles:
            try:
                tests_passed, test_error = run_tests(full_solution, test_code, entry_point=entry_point, timeout=10)
                if tests_passed:
                    test_pass_count += 1
                else:
                    logger.debug(f"Tests failed for {task_id}: {test_error}")
            except Exception as e:
                logger.debug(f"Error running tests for {task_id}: {e}")

        total_evaluated += 1

        # Log progress periodically
        if (idx + 1) % 10 == 0:
            current_compile_rate = (compile_count / total_evaluated) * 100
            current_test_rate = (test_pass_count / total_evaluated) * 100
            logger.info(
                f"Progress: {idx + 1}/{total_samples} | "
                f"Compile rate: {current_compile_rate:.2f}% | "
                f"Test pass rate: {current_test_rate:.2f}%"
            )

    # Calculate final metrics
    compile_rate = (compile_count / total_evaluated) * 100 if total_evaluated > 0 else 0.0
    test_pass_rate = (test_pass_count / total_evaluated) * 100 if total_evaluated > 0 else 0.0

    return {
        "compile_rate": compile_rate,
        "test_pass_rate": test_pass_rate,
        "compile_count": compile_count,
        "test_pass_count": test_pass_count,
        "total_evaluated": total_evaluated
    }


def main():
    """Main evaluation function."""
    # Replace argparse with direct assignments for Colab
    model_path = "/content/drive/MyDrive/PyPilot/outputs/qwen-lora/final"
    base_model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
    data_dir = "/content/drive/MyDrive/PyPilot/data/leetcode"
    max_samples = 50
    use_4bit = False
    max_new_tokens = 768
    temperature = 0.2

    # Load model
    model, tokenizer = load_lora_model(base_model_id, model_path, use_4bit=use_4bit)

    # Load dataset
    logger.info(f"Loading dataset from {data_dir}...")
    dataset = load_from_disk(data_dir)
    logger.info(f"Dataset loaded. Test set size: {len(dataset['test'])}")

    # Evaluate
    metrics = evaluate_on_dataset(
        model,
        tokenizer,
        dataset,
        max_samples=max_samples,
        include_prelude=True,  # Match training setting
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    # Log final results
    logger.info("=" * 60)
    logger.info("FINAL EVALUATION RESULTS")
    logger.info("=" * 60)
    logger.info(f"Model path: {model_path}")
    logger.info(f"Base model: {base_model_id}")
    logger.info(f"Total samples evaluated: {metrics['total_evaluated']}")
    logger.info(f"Compilation rate: {metrics['compile_rate']:.2f}% ({metrics['compile_count']}/{metrics['total_evaluated']})")
    logger.info(f"Test pass rate: {metrics['test_pass_rate']:.2f}% ({metrics['test_pass_count']}/{metrics['total_evaluated']})")
    logger.info("=" * 60)

    print("=" * 60)
    print("FINAL EVALUATION RESULTS")
    print("=" * 60)
    print(f"Model path: {model_path}")
    print(f"Base model: {base_model_id}")
    print(f"Total samples evaluated: {metrics['total_evaluated']}")
    print(f"Compilation rate: {metrics['compile_rate']:.2f}% ({metrics['compile_count']}/{metrics['total_evaluated']})")
    print(f"Test pass rate: {metrics['test_pass_rate']:.2f}% ({metrics['test_pass_count']}/{metrics['total_evaluated']})")
    print("=" * 60)


main()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path='Qwen/Qwen2.5-Coder-7B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'k_proj', 'q_proj', 'v_proj', 'up_proj', 'o_proj', 'down_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)}




Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Prompt: <|im_start|>user
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
You are given an integer n and a 2D integer array queries.
There are n cities numbered from 0 to n - 1. Initially, there is a unidirectional road from city i to city i + 1 for all 0 <= i < n - 1.
queries[i] = [ui, vi] represents the addition of a new unidirectional road from city ui to city vi. After each query, you need to find the length of the shortest path from city 0 to city n - 1.
Return an array answer where for each i in the range [0, queries.length - 1], answer[i] is the length of the shortest path from city 0 to city n - 1 after processing the first i + 1 queries.
 
Example 1:

Input: n = 5, queries = [[2,4],[0,2],[0,4]]
Output: [3,2,1]
Explanation: 

After the addition of the road from 2 to 4, the length of the shortest path from 0 to 4 is 3.

After the



Evaluating:   2%|▏         | 1/50 [00:13<10:48, 13.23s/it]

...


Generated Completion: class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        d = {"UP": -n, "RIGHT": 1, "DOWN": n, "LEFT": -1}
        i = 0
        for c in commands:
            i += d[c]
        return i

Prompt: <|im_start|>user
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
There is an undirected tree with n nodes labeled from 0 to n - 1, and rooted at node 0. You are given a 2D integer array edges of length n - 1, where edges[i] = [ai, bi] indicates that there is an edge between nodes ai and bi in the tree.
A node is good if all the subtrees rooted at its children have the same size.
Return the number of good nodes in the given tree.
A subtree of treeName is a tree consisting of a node in treeName and all of its descendants.
 
Example 1:

Input: edges = [[0,1],[0,2],[1,3],[1,4],[2,5],[2,



Evaluating:  10%|█         | 5/50 [01:02<08:49, 11.76s/it]


Generated Completion: class Solution:
    def countGoodNodes(self, edges: List[List[int]]) -> int:
        g = defaultdict(list)
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)

        def dfs(a: int, fa: int) -> int:
            cnt = 1
            for b in g[a]:
                if b != fa:
                    t = dfs(b, a)
                    nonlocal ans
                    if t == mx:
                        ans += cnt
                    elif t > mx:
                        ans = cnt
                        mx = t
                    cnt += t
            return cnt

        ans = mx = 0
        dfs(0, -1)
        return ans + 1





Evaluating:  12%|█▏        | 6/50 [01:20<10:11, 13.89s/it]

Evaluating:  14%|█▍        | 7/50 [01:36<10:23, 14.51s/it]

Evaluating:  16%|█▌        | 8/50 [01:48<09:26, 13.49s/it]

Evaluating:  18%|█▊        | 9/50 [01:56<08:07, 11.88s/it]

Evaluating:  20%|██        | 10/50 [02:06<07:38, 11.45s/it]

Evaluating:  22%|██▏       | 11/50 [02:13<06:30, 10.02s/it]

Evaluating:  24%|██▍       | 12/50 [02:20<05:43,  9.04s/it]

Evaluating:  26%|██▌       | 13/50 [02:27<05:16,  8.55s/it]

Evaluating:  28%|██▊       | 14/50 [02:34<04:44,  7.91s/it]

Evaluating:  30%|███       | 15/50 [02:43<04:51,  8.33s/it]

Evaluating:  32%|███▏      | 16/50 [02:58<05:46, 10.20s/it]

Evaluating:  34%|███▍      | 17/50 [03:03<04:44,  8.61s/it]

Evaluating:  36%|███▌      | 18/50 [03:12<04:45,  8.94s/it]

Evaluating:  38%|███▊      | 19/50 [03:20<04:28,  8.65s/it]

Evaluating:  40%|████      | 20/50 [03:30<04:29,  8.99s/it]

Evaluating:  42%|████▏     | 21/50 [03:36<03:51,  7.98s/it]

Evaluating:  44%|████▍    

FINAL EVALUATION RESULTS
Model path: /content/drive/MyDrive/PyPilot/outputs/qwen-lora/final
Base model: Qwen/Qwen2.5-Coder-7B-Instruct
Total samples evaluated: 50
Compilation rate: 100.00% (50/50)
Test pass rate: 14.00% (7/50)


In [10]:
import numpy as np
from tqdm import tqdm

def analyze_dataset_lengths(dataset, tokenizer, max_seq_length=1024, min_supervised_tokens=64):
    """
    Analyze how many examples will be truncated or poorly supervised.
    """

    prompt_lengths = []
    response_lengths = []
    total_lengths = []
    supervised_after_trunc = []

    for example in tqdm(dataset):
        prompt_ids = tokenizer(
            example["prompt"],
            add_special_tokens=False
        )["input_ids"]

        response_ids = tokenizer(
            example["response"],
            add_special_tokens=False
        )["input_ids"]

        prompt_len = len(prompt_ids)
        response_len = len(response_ids)
        total_len = prompt_len + response_len

        # simulate truncation
        truncated_total = min(total_len, max_seq_length)

        # supervised tokens are response tokens that survive truncation
        supervised_tokens = max(
            0,
            truncated_total - prompt_len
        )

        prompt_lengths.append(prompt_len)
        response_lengths.append(response_len)
        total_lengths.append(total_len)
        supervised_after_trunc.append(supervised_tokens)

    prompt_lengths = np.array(prompt_lengths)
    response_lengths = np.array(response_lengths)
    total_lengths = np.array(total_lengths)
    supervised_after_trunc = np.array(supervised_after_trunc)

    print("\n========== DATASET LENGTH ANALYSIS ==========\n")

    print(f"Total examples: {len(dataset)}")
    print(f"Max sequence length: {max_seq_length}\n")

    print("---- Prompt Length ----")
    print(f"Mean: {prompt_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(prompt_lengths, 95):.1f}")
    print(f"Max: {prompt_lengths.max()}\n")

    print("---- Response Length ----")
    print(f"Mean: {response_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(response_lengths, 95):.1f}")
    print(f"Max: {response_lengths.max()}\n")

    print("---- Total Length ----")
    print(f"Mean: {total_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(total_lengths, 95):.1f}")
    print(f"Max: {total_lengths.max()}\n")

    too_long = (total_lengths > max_seq_length).sum()
    print(f"Examples exceeding max_seq_length: {too_long} "
          f"({100*too_long/len(dataset):.2f}%)")

    no_supervision = (supervised_after_trunc == 0).sum()
    print(f"Examples with ZERO supervised tokens after truncation: "
          f"{no_supervision} ({100*no_supervision/len(dataset):.2f}%)")

    low_supervision = (supervised_after_trunc < min_supervised_tokens).sum()
    print(f"Examples with <{min_supervised_tokens} supervised tokens: "
          f"{low_supervision} ({100*low_supervision/len(dataset):.2f}%)")

    print("\n=============================================\n")

    return {
        "prompt_lengths": prompt_lengths,
        "response_lengths": response_lengths,
        "total_lengths": total_lengths,
        "supervised_after_trunc": supervised_after_trunc,
    }

data_dir = "/content/drive/MyDrive/PyPilot/data/leetcode"
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    padding_side="right",
)

# Load dataset
logger.info(f"Loading dataset from {data_dir}...")
dataset = load_from_disk(data_dir)
analysis = analyze_dataset_lengths(
    dataset["train"],
    tokenizer,
    max_seq_length=3072,
    min_supervised_tokens=64,
)



  0%|          | 0/2641 [00:00<?, ?it/s]

  1%|▏         | 37/2641 [00:00<00:07, 369.61it/s]

  3%|▎         | 76/2641 [00:00<00:06, 376.05it/s]

  4%|▍         | 114/2641 [00:00<00:06, 374.57it/s]

  6%|▌         | 153/2641 [00:00<00:06, 377.52it/s]

  7%|▋         | 191/2641 [00:00<00:06, 376.52it/s]

  9%|▊         | 229/2641 [00:00<00:06, 375.40it/s]

 10%|█         | 267/2641 [00:00<00:06, 369.50it/s]

 12%|█▏        | 304/2641 [00:00<00:06, 367.90it/s]

 13%|█▎        | 342/2641 [00:00<00:06, 369.14it/s]

 14%|█▍        | 380/2641 [00:01<00:06, 369.92it/s]

 16%|█▌        | 417/2641 [00:01<00:06, 363.54it/s]

 17%|█▋        | 455/2641 [00:01<00:05, 365.56it/s]

 19%|█▊        | 492/2641 [00:01<00:05, 365.27it/s]

 20%|██        | 530/2641 [00:01<00:05, 367.16it/s]

 21%|██▏       | 567/2641 [00:01<00:05, 365.69it/s]

 23%|██▎       | 604/2641 [00:01<00:05, 361.23it/s]

 24%|██▍       | 641/2641 [00:01<00:05, 359.50it/s]

 26%|██▌       | 677/2641 [00:01<00:05, 355.46it/s]

 27%


========== DATASET LENGTH ANALYSIS ==========

Total examples: 2641
Max sequence length: 3072

---- Prompt Length ----
Mean: 438.5
95th percentile: 440.0
Max: 453

---- Response Length ----
Mean: 337.7
95th percentile: 692.0
Max: 2048

---- Total Length ----
Mean: 776.2
95th percentile: 1132.0
Max: 2488

Examples exceeding max_seq_length: 0 (0.00%)
Examples with ZERO supervised tokens after truncation: 0 (0.00%)
Examples with <64 supervised tokens: 14 (0.53%)




In [ ]:
!pip install -U bitsandbytes>=0.46.1


In [ ]:
# ============================================================
# Evaluate BASE Qwen2.5-Coder (NO LoRA)
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_from_disk
from tqdm import tqdm
import logging

# -----------------------
# CONFIG
# -----------------------
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATA_DIR = "/content/drive/MyDrive/PyPilot/data/leetcode"
USE_4BIT = False
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
MAX_SAMPLES = 350   # set to 50 for quick test

# -----------------------
# Logging
# -----------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -----------------------
# Chat tokens (must match training format)
# -----------------------
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"

# -----------------------
# Prompt builder
# -----------------------
def build_model_prompt(row: dict) -> str:
    user_content = (row.get("query") or "").strip()
    user_content = user_content.replace("(use the provided format with backticks)", "")
    user_content = user_content.replace("and enclose your code within delimiters.", "")
    user_content += "\n\nRespond with only the Python code. No explanations, no markdown."
    return f"{CHAT_USER}{user_content}{CHAT_END}\n{CHAT_ASSISTANT}"

# -----------------------
# Generation
# -----------------------
def generate_code(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    eos_ids = []
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            eos_ids.append(tid)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_ids
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    for marker in ["<|im_end|>", "<|endoftext|>"]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]

    return text.strip()

# -----------------------
# Import your harness utilities
# (These must already exist in your notebook)
# -----------------------
# check_compilation
# extract_code_from_completion
# run_tests

# -----------------------
# Evaluation
# -----------------------
def evaluate_base_model():

    logger.info("Loading base model...")

    bnb_config = None
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        padding_side="right",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)

    dataset = load_from_disk(DATA_DIR)
    test_split = dataset["test"]

    total_samples = len(test_split) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(test_split))

    compile_count = 0
    test_pass_count = 0

    for idx in tqdm(range(total_samples), desc="Evaluating Base Model"):

        row = test_split[idx]

        prompt = build_model_prompt(row)
        starter_code = row.get("starter_code", "")
        test_code = row.get("test", "")
        entry_point = row.get("entry_point", "candidate")

        if not prompt.strip() or not test_code:
            continue

        try:
            generated = generate_code(model, tokenizer, prompt)
            if idx < 5:
                print(f"\nGenerated Completion: {generated}\n")
        except Exception:
            continue

        extracted = extract_code_from_completion(generated, starter_code)

        compiles, _ = check_compilation(extracted)

        if compiles:
            compile_count += 1
            passed, _ = run_tests(extracted, test_code, entry_point=entry_point)
            if passed:
                test_pass_count += 1

    compile_rate = 100 * compile_count / total_samples
    pass_rate = 100 * test_pass_count / total_samples

    print("=" * 60)
    print("BASE MODEL RESULTS")
    print("=" * 60)
    print(f"Total evaluated: {total_samples}")
    print(f"Compile rate: {compile_rate:.2f}% ({compile_count}/{total_samples})")
    print(f"Test pass rate: {pass_rate:.2f}% ({test_pass_count}/{total_samples})")
    print("=" * 60)


# Run evaluation
evaluate_base_model()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Evaluating Base Model:   0%|          | 1/228 [00:07<29:07,  7.70s/it]


Generated Completion: ```python
from collections import defaultdict
import heapq

class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
        graph = defaultdict(list)
        for u, v in queries:
            graph[u].append((v, 1))
        
        dist = [float('inf')] * n
        dist[0] = 0
        
        pq = [(0, 0)]
        while pq:
            d, node = heapq.heappop(pq)
            if d > dist[node]:
                continue
            for neighbor, weight in graph[node]:
                new_dist = d + weight
                if new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
        
        result = []
        current_min = float('inf')
        for d in dist:
            current_min = min(current_min, d)
            result.append(current_min)
        
        return result
```




Evaluating Base Model:   1%|          | 2/228 [00:21<43:10, 11.46s/it]


Generated Completion: ```python
from collections import defaultdict
import heapq

class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
        graph = defaultdict(list)
        for u, v in queries:
            graph[u].append((v, 1))
            if v > 0:
                graph[v-1].append((v, 1))
        
        dist = [float('inf')] * n
        dist[0] = 0
        
        pq = [(0, 0)]
        while pq:
            d, node = heapq.heappop(pq)
            if d > dist[node]:
                continue
            for neighbor, weight in graph[node]:
                new_dist = d + weight
                if new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
        
        result = []
        for i in range(len(queries)):
            result.append(dist[n-1])
            if i + 1 < len(queries):
                u, v = queries[i+1]
                if u >

...


Generated Completion: ```python
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        x, y = 0, 0
        for cmd in commands:
            if cmd == "UP":
                y -= 1
            elif cmd == "DOWN":
                y += 1
            elif cmd == "LEFT":
                x -= 1
            elif cmd == "RIGHT":
                x += 1
        return x * n + y
```




Evaluating Base Model:   2%|▏         | 5/228 [00:34<22:09,  5.96s/it]


Generated Completion: ```python
from collections import defaultdict

class Solution:
    def countGoodNodes(self, edges: List[List[int]]) -> int:
        graph = defaultdict(list)
        for u, v in edges:
            graph[u].append(v)
            graph[v].append(u)
        
        self.count = 0
        
        def dfs(node, parent):
            sizes = []
            for neighbor in graph[node]:
                if neighbor != parent:
                    sizes.append(dfs(neighbor, node))
            if not sizes or len(set(sizes)) == 1:
                self.count += 1
            return 1 + sum(sizes)
        
        dfs(0, -1)
        return self.count
```




Evaluating Base Model: 100%|██████████| 228/228 [20:57<00:00,  5.51s/it]

BASE MODEL RESULTS
Total evaluated: 228
Compile rate: 100.00% (228/228)
Test pass rate: 14.04% (32/228)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
# Quick generation diagnostic
test_prompt = "<|im_start|>user\nWrite a python function to add two numbers.<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

print(f"Input token count: {inputs['input_ids'].shape[1]}")

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
print(f"<|im_end|> token id: {im_end_id}")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=[tokenizer.eos_token_id, im_end_id],
)

print(f"Output token count: {outputs.shape[1]}")
print(f"New tokens generated: {outputs.shape[1] - inputs['input_ids'].shape[1]}")
print("\n--- Full raw output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))
print("\n--- New tokens only ---")
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False))

NameError: name 'AutoTokenizer' is not defined

In [ ]:
import os
from pathlib import Path

# 1. Inspect the model directory
model_path = Path("/content/drive/MyDrive/PyPilot/outputs/qwen-lora/final")
print(f"Checking model directory: {model_path}")

if model_path.exists():
    print("Files found:")
    for file in model_path.glob("*"):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - {file.name}: {size_mb:.2f} MB")
else:
    print("❌ Model directory does not exist! The training likely failed completely.")


Checking model directory: /content/drive/MyDrive/PyPilot/outputs/qwen-lora/final
Files found:
  - tokenizer.json: 10.89 MB
  - training_args.bin: 0.01 MB
  - chat_template.jinja: 0.00 MB
  - tokenizer_config.json: 0.00 MB
  - README.md: 0.00 MB
  - adapter_config.json: 0.00 MB
  - adapter_model.safetensors: 77.05 MB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# 2. Run a generation sanity check
# We'll use a simple prompt to see if the model behaves like a code model

try:
    base_model_id = "Qwen/Qwen2.5-Coder-7B"
    adapter_path = str(model_path)

    print(f"\nLoading base model: {base_model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)

    # Load in FP16 to avoid the bitsandbytes error for this diagnostic
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    print(f"Loading adapter from {adapter_path}...")
    model = PeftModel.from_pretrained(model, adapter_path)

    # Test Prompt
    test_prompt = "<|im_start|>user\nWrite a python function to add two numbers.<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

    print("Generating...")
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    print("\n" + "="*40)
    print("RAW MODEL OUTPUT:")
    print("="*40)
    print(tokenizer.decode(outputs[0], skip_special_tokens=False))
    print("="*40)

except Exception as e:
    print(f"\n❌ Diagnostic failed: {e}")


Loading base model: Qwen/Qwen2.5-Coder-7B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

Loading adapter from /content/drive/MyDrive/PyPilot/outputs/qwen-lora/final...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generating...

RAW MODEL OUTPUT:
<|im_start|>user
Write a python function to add two numbers.<|im_end|>
<|im_start|>assistant
Here is a simple Python function that adds two numbers:

```python
def add_numbers(a, b):
    return a + b
```

You can use this function by calling it with two numbers as arguments, like this:

```python
result = add_numbers(3, 5)
print(result)  # Output: 8
```<|endoftext|>
